# Preprocessing

Feature engineering for the Uber Dynamic Pricing dataset.

- Encodes categorical columns
- Adds demand/supply ratio + surge flag
- Scales features to [0, 1] with MinMaxScaler
- Saves `X.npy`, `y.npy`, `scaler.pkl` to `data/processed/`

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
RAW_PATH      = os.path.join(PROJECT_ROOT, "data/raw/dynamic_pricing_synth.csv")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data/processed")

print(f"Raw data : {RAW_PATH}")
print(f"Output   : {PROCESSED_DIR}")

Raw data : /home/lux/luxchar/Uber-Dynamic-Pricing/data/raw/dynamic_pricing_synth.csv
Output   : /home/lux/luxchar/Uber-Dynamic-Pricing/data/processed


## Load raw data

In [4]:
df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (21000, 10)


,Number_of_Riders,Number_of_Drivers,Location_Category,Customer_Loyalty_Status,Number_of_Past_Rides,Average_Ratings,Time_of_Booking,Vehicle_Type,Expected_Ride_Duration,Historical_Cost_of_Ride
0,90,45,Urban,Silver,13,4.47,Night,Premium,90,284.257273
1,58,39,Suburban,Silver,72,4.06,Evening,Economy,43,173.874753
2,42,31,Rural,Silver,0,3.99,Afternoon,Premium,76,329.795469


In [5]:
df.describe()

,Number_of_Riders,Number_of_Drivers,Number_of_Past_Rides,Average_Ratings,Expected_Ride_Duration,Historical_Cost_of_Ride
count,21000.000000,21000.000000,21000.000000,21000.000000,21000.000000,21000.000000
mean,59.657619,27.704571,50.073524,4.255215,99.726857,374.268495
std,22.765532,17.865197,27.180514,0.405600,45.941984,173.942991
min,1.000000,1.000000,0.000000,3.500000,1.000000,25.993449
25%,44.000000,14.000000,30.000000,3.960000,67.000000,246.922016
50%,60.000000,27.000000,50.000000,4.260000,100.000000,377.253884
75%,76.000000,40.000000,70.000000,4.560000,134.000000,501.870473
max,100.000000,89.000000,100.000000,5.000000,180.000000,836.116419


## Encode categoricals

In [6]:
LOCATION_MAP = {"Urban": 0, "Suburban": 1, "Rural": 2}
LOYALTY_MAP  = {"Regular": 0, "Silver": 1, "Gold": 2}
TIME_MAP     = {"Morning": 0, "Afternoon": 1, "Evening": 2, "Night": 3}
VEHICLE_MAP  = {"Economy": 0, "Premium": 1}

df["Location_Category"]      = df["Location_Category"].map(LOCATION_MAP)
df["Customer_Loyalty_Status"] = df["Customer_Loyalty_Status"].map(LOYALTY_MAP)
df["Time_of_Booking"]         = df["Time_of_Booking"].map(TIME_MAP)
df["Vehicle_Type"]            = df["Vehicle_Type"].map(VEHICLE_MAP)

df.head(3)

,Number_of_Riders,Number_of_Drivers,Location_Category,Customer_Loyalty_Status,Number_of_Past_Rides,Average_Ratings,Time_of_Booking,Vehicle_Type,Expected_Ride_Duration,Historical_Cost_of_Ride
0,90,45,0,1,13,4.47,3,1,90,284.257273
1,58,39,1,1,72,4.06,2,0,43,173.874753
2,42,31,2,1,0,3.99,1,1,76,329.795469


## Feature engineering

In [7]:
df["demand_supply_ratio"] = df["Number_of_Riders"] / (df["Number_of_Drivers"] + 1e-6)
df["is_surge"] = (df["demand_supply_ratio"] > df["demand_supply_ratio"].median()).astype(int)

print(f"Surge rate: {df['is_surge'].mean():.1%}")
print(f"DSR range : [{df['demand_supply_ratio'].min():.2f}, {df['demand_supply_ratio'].max():.2f}]")

Surge rate: 50.0%
DSR range : [0.01, 100.00]


## Scale & save

In [8]:
STATE_COLS = [
    "Number_of_Riders",
    "Number_of_Drivers",
    "demand_supply_ratio",
    "is_surge",
    "Location_Category",
    "Customer_Loyalty_Status",
    "Number_of_Past_Rides",
    "Average_Ratings",
    "Time_of_Booking",
    "Vehicle_Type",
    "Expected_Ride_Duration",
]

scaler = MinMaxScaler()
X = scaler.fit_transform(df[STATE_COLS])
y = df["Historical_Cost_of_Ride"].values

print(f"X shape : {X.shape}")
print(f"y range : [{y.min():.1f}, {y.max():.1f}]")
print(f"Features: {STATE_COLS}")

X shape : (21000, 11)
y range : [26.0, 836.1]
Features: ['Number_of_Riders', 'Number_of_Drivers', 'demand_supply_ratio', 'is_surge', 'Location_Category', 'Customer_Loyalty_Status', 'Number_of_Past_Rides', 'Average_Ratings', 'Time_of_Booking', 'Vehicle_Type', 'Expected_Ride_Duration']


In [9]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
np.save(os.path.join(PROCESSED_DIR, "X.npy"), X)
np.save(os.path.join(PROCESSED_DIR, "y.npy"), y)
joblib.dump(scaler, os.path.join(PROCESSED_DIR, "scaler.pkl"))
print(f"Saved to {PROCESSED_DIR}")

Saved to /home/lux/luxchar/Uber-Dynamic-Pricing/data/processed
